In [1]:
# 6. Cross-validation of LLM annotation and variant list

import re
from pathlib import Path

from gensim.models import FastText
import pandas as pd

DATA = Path("../data")   # not included in the repository
TAB = Path("../tab")

GENERAL = ["making noise", "disrupting nightrest"]
PRACTICES = ["yelling", "singing", "arguing", "scolding", "playing inn music",
             "playing street music", "doorbell ringing", "traffic noise"]

In [2]:
# Definitions

FIRST_YEAR, LAST_YEAR = 1876, 1939


NIGHT = ["nacht", "avondschemering", "ochtendschemering"]   # night incl. evening/morning twilight


# Seed forms of the variant list: five noise terms, their "nacht-" compounds, and "nachtrust".
SEEDS = ["lawijd", "lawaai", "gerucht", "kabaal", "rumoer",
         "nachtlawijd", "nachtlawaai", "nachtrumoer", "nachtgerucht", "nachtrust"]


def event_id(df):
    """Unique incident key: register + incident number."""
    number = pd.to_numeric(df["Incident Number"], errors="coerce").astype("Int64").astype(str)
    return df["archive_file"].astype(str).str.strip() + "_" + number


def load_incidents(folder, columns=("archive_file", "Incident Number", "year", "district", "astro", "incident_text"),
                   one_row=True):
    """All incidents of all districts. The tables have one row per person involved; with one_row=True
    only the first row of each incident is kept."""
    files = [f for f in sorted(folder.glob("*.xlsx")) if not f.name.startswith(("~$", "._"))]
    df = pd.concat([pd.read_excel(f, usecols=list(columns)) for f in files], ignore_index=True)
    df["event_id"] = event_id(df)
    return (df.drop_duplicates("event_id") if one_row else df).reset_index(drop=True)


def word_set(text):
    """Words of an incident description as used for lexical retrieval."""
    if not isinstance(text, str):
        return set()
    return {re.sub(r"[^\w]", "", w) for w in text.lower().split()}


def variant_groups(fasttext_model, threshold=0.75, topn=300):
    """FastText neighbours of each seed with cosine >= threshold. Neighbours with a
    "nacht-" prefix are only kept for seeds that carry that prefix themselves, and vice versa."""
    groups = {}
    for seed in SEEDS:
        forms = {seed}
        for word, score in fasttext_model.wv.most_similar(seed, topn=topn):
            if score >= threshold and word.startswith("nacht") == seed.startswith("nacht"):
                forms.add(word)
        groups[seed] = forms
    return groups

In [3]:
# Inputs

llm = pd.read_excel(DATA / "llm" / "geluidsoverlast_llm.xlsx")
llm = llm[llm["astronomical_category"].isin(NIGHT)].assign(event_id=event_id)
other_labels = [c for c in llm.columns if c in PRACTICES + ["other_sound"]]
llm = llm.groupby("event_id")[GENERAL + other_labels].max().astype(bool)
llm_general = set(llm.index[llm[GENERAL].any(axis=1)])
print(f"LLM, general nuisance label: {len(llm_general)} "
      f"(both labels: {llm.loc[list(llm_general), GENERAL].all(axis=1).sum()})")

matches = pd.read_csv(TAB / "variant_list_matches.csv")
matches = matches[(matches["district"] == 3) & matches["night"]]
variant = set(matches["event_id"])
print(f"variant list: {len(variant)}")

final = pd.read_excel(DATA / "geluidsoverlast_w3_nacht_allepersonen.xlsx")
final = final[final["archive_file"].notna()].assign(event_id=event_id)
final_ids = set(final["event_id"])
print(f"final dataset: {len(final_ids)}")

LLM, general nuisance label: 740 (both labels: 122)
variant list: 742


final dataset: 818


In [4]:
# Overlap

overlap = llm_general & variant
variant_only = variant - llm_general
print(f"overlap: {len(overlap)} | LLM also found by list: {len(overlap) / len(llm_general):.1%} "
      f"| list also found by LLM: {len(overlap) / len(variant):.1%}")

kept = variant_only & final_ids
with_other_label = {e for e in kept if e in llm.index and llm.loc[e, other_labels].any()}
print(f"variant-list only: {len(variant_only)}")
print(f"  kept: {len(kept)} ({len(with_other_label)} with another LLM sound label, "
      f"{len(kept - with_other_label)} after manual check)")
print(f"  false positives: {len(variant_only - final_ids)}")
print(f"found by both methods but removed after inspection: {sorted(overlap - final_ids)}")

overlap: 594 | LLM also found by list: 80.3% | list also found by LLM: 80.1%
variant-list only: 148
  kept: 80 (58 with another LLM sound label, 22 after manual check)
  false positives: 68
found by both methods but removed after inspection: ['731#1595_755', 'MA#17618_4']


In [5]:
genuine = variant & final_ids
print(f"precision of the variant list: {len(genuine)}/{len(variant)} = {len(genuine) / len(variant):.1%}")
print(f"coverage of the final dataset: {len(genuine)}/{len(final_ids)} = {len(genuine) / len(final_ids):.1%}")
print(f"found only through the LLM:    {len(final_ids - variant)}")
assert final_ids == (llm_general | kept) - (overlap - final_ids)

precision of the variant list: 672/742 = 90.6%
coverage of the final dataset: 672/818 = 82.2%
found only through the LLM:    146


In [6]:
# gerucht: share of noise complaints among the incidents that contain the word itself

with_gerucht = {e for e, m in zip(matches["event_id"], matches["matched"])
                if {"gerucht", "nachtgerucht"} & set(m.split("; "))}
print(f"noise complaint: {len(with_gerucht & final_ids)} | not a noise complaint: {len(with_gerucht - final_ids)}")

noise complaint: 37 | not a noise complaint: 31


In [7]:
# Share of noise complaints that also carry a specific sound practice.

practice = final.groupby("event_id")[PRACTICES].max().fillna(False).astype(bool).any(axis=1)
print(f"with a specific practice: {practice.sum()} ({practice.mean():.1%})")

with a specific practice: 492 (60.1%)


In [8]:
# Prevalence

incidents = load_incidents(DATA / "incidents")
night3 = incidents[(incidents["district"] == 3) & incidents["astro"].isin(NIGHT)
                   & incidents["year"].between(FIRST_YEAR, LAST_YEAR)]
print(f"nocturnal incidents, district 3: {len(night3):,}")
print(f"noise complaints within this set: {len(final_ids & set(night3['event_id']))} of {len(final_ids)}")
print(f"prevalence: {len(final_ids) / len(night3):.1%}")

nocturnal incidents, district 3: 15,069
noise complaints within this set: 818 of 818
prevalence: 5.4%


In [9]:
# Sensitivity to the FastText threshold (§5.2, footnote)

ft_rare = FastText.load(str(DATA / "models" / "FastText_htr.model"))
words = incidents.loc[(incidents["district"] == 3) & incidents["astro"].isin(NIGHT)].set_index("event_id")["incident_text"].map(word_set)

rows = []
for threshold in [0.65, 0.70, 0.75, 0.80, 0.85]:
    forms = set().union(*variant_groups(ft_rare, threshold).values())
    found = set(words.index[words.map(lambda ws: not ws.isdisjoint(forms))])
    true, unchecked = found & final_ids, found - final_ids - variant
    rows.append({"threshold": threshold, "forms": len(forms), "retrieved": len(found), "in_final": len(true),
                 "unchecked": len(unchecked), "precision_min": round(len(true) / len(found), 3),
                 "precision_max": round((len(true) + len(unchecked)) / len(found), 3),
                 "coverage": round(len(true) / len(final_ids), 3)})
thresholds = pd.DataFrame(rows)
thresholds.to_csv(TAB / "threshold_sensitivity.csv", index=False)
thresholds

,threshold,forms,retrieved,in_final,unchecked,precision_min,precision_max,coverage
0,0.65,1125,1443,705,668,0.489,0.951,0.862
1,0.70,911,1092,695,327,0.636,0.936,0.850
2,0.75,695,742,672,0,0.906,0.906,0.822
3,0.80,501,584,532,0,0.911,0.911,0.650
4,0.85,376,544,499,0,0.917,0.917,0.610


In [10]:
# Coverage of the variant list by period

FRENCH = re.compile(r"tapage|bruit|nocturne|tumulte|vacarme|cris|chant")
texts = incidents.set_index("event_id")["incident_text"].fillna("").str.lower()

per_incident = final.drop_duplicates("event_id").set_index("event_id")[["year"]]
per_incident["found_by_list"] = per_incident.index.isin(variant)
per_incident["french"] = [bool(FRENCH.search(texts.get(e, ""))) for e in per_incident.index]
per_incident["period"] = pd.cut(per_incident["year"], [1876, 1891, 1914, 1939],
                                labels=["1877-1891", "1912-1914", "1919-1939"])

coverage = per_incident.groupby("period", observed=True).agg(
    complaints=("found_by_list", "size"), found_by_list=("found_by_list", "sum"))
coverage["coverage"] = (coverage["found_by_list"] / coverage["complaints"]).round(3)
missed = per_incident[~per_incident["found_by_list"]]
coverage["missed"] = coverage["complaints"] - coverage["found_by_list"]
coverage["missed_french"] = missed.groupby("period", observed=True)["french"].sum()
coverage.to_csv(TAB / "variant_list_coverage_by_period.csv")
coverage

,complaints,found_by_list,coverage,missed,missed_french
period,,,,,
1877-1891,502,404,0.805,98,30
1912-1914,91,85,0.934,6,0
1919-1939,225,183,0.813,42,2
